# 💼 Job Salary Analysis — End-to-End Data Science Project

| | |
|---|---|
| **Author** | A Sarika |
| **Program** | IBM SkillsBuild Data Analytics with AI — Academic Internship (AICTE) |

---

> **Dataset:** `Salary_Data.csv` · 6,698 clean records · 6 raw features  
> **Goal:** Predict employee salary using regression models and engineered features  
> **Best Model:** Gradient Boosting · Test R² = **0.8975** · Test MAE = **\$7,693**

---

## Notebook Structure

| Cell | Phase | Description |
|------|-------|-------------|
| 1 | Setup & Data Loading | Imports, load CSV, inspect shape/types/missing values |
| 2 | Exploratory Data Analysis | Descriptive stats, unique values, percentile summary |
| 3 | Feature Engineering | Encoding, interaction terms, log-transform target |
| 4 | Visualizations | 8 inline charts covering distributions, correlations, top roles |
| 5 | Model Training & Evaluation | 4 models, 5-fold CV, benchmark table, feature importances |

---
## Cell 1 — Setup & Data Loading

Load all libraries, read `Salary_Data.csv`, inspect the raw dataframe shape, column data types, and the count/percentage of missing values. Drop the 6 rows with any `NaN` and standardise column names to lowercase with underscores.

In [ ]:
import os
import warnings
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_csv('Salary_Data.csv')
print(f'Raw shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns  : {list(df.columns)}')
print()

# Data types
print('── Data Types ──────────────────────────────')
print(df.dtypes)

# Missing values
print('\n── Missing Values ──────────────────────────')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.concat([missing, missing_pct], axis=1, keys=['Count', '%']))

# Clean: drop NaN rows, standardise column names
df = df.dropna().reset_index(drop=True)
df.columns = pd.Index([c.strip().lower().replace(' ', '_') for c in df.columns])

print(f'\nClean shape   : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Column names  : {list(df.columns)}')
df.head()

---
## Cell 2 — Exploratory Data Analysis (EDA)

Profile each column for unique-value cardinality and generate a full descriptive statistics table. Display the salary percentile distribution to understand the target variable's shape and spread before any transformation.

In [ ]:
# ── Unique-value audit ───────────────────────────────────────────────────────
print('── Unique Values per Column ────────────────')
for col in df.columns:
    print(f'  {col:25s}: {df[col].nunique():>4} unique')

# ── Descriptive statistics ───────────────────────────────────────────────────
print('\n── Descriptive Statistics ──────────────────')
display(df.describe(include='all').T)

# ── Salary percentile summary ────────────────────────────────────────────────
print('\n── Salary Percentile Summary ───────────────')
pct = df['salary'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
display(pct.to_frame(name='Salary (USD)').style.format('${:,.0f}', subset=['Salary (USD)']
    if pct.index.isin(['min','max','mean','std','10%','25%','50%','75%','90%','95%','99%']).any()
    else None))

# ── Education level value counts ─────────────────────────────────────────────
print('\n── Education Level Value Counts ─────────────')
display(df['education_level'].value_counts().to_frame())

# ── Top 10 most common job titles ────────────────────────────────────────────
print('\n── Top 10 Job Titles ────────────────────────')
display(df['job_title'].value_counts().head(10).to_frame())

---
## Cell 3 — Feature Engineering

Transform raw columns into a 24-feature matrix ready for scikit-learn:

| Feature | Method |
|---|---|
| `education_level_clean` | Fuzzy ordinal mapping 1–4 (handles all 7 text variants) |
| `gender_encoded` | `LabelEncoder` → 0 / 1 / 2 |
| `is_senior` | Keyword flag: *senior, director, manager, vp, chief, head, lead, principal* |
| `jobd_*` (16 cols) | One-hot dummies: top-15 job titles + 'Other' |
| `exp_x_edu` | Interaction: `years_of_experience × education_level_clean` |
| `age_squared` | Polynomial: `age²` |
| `exp_squared` | Polynomial: `years_of_experience²` |
| `log_salary` | `log1p(salary)` — normalises right-skewed target |

In [ ]:
# ── Education ordinal mapping ────────────────────────────────────────────────
def map_education(val: object) -> Optional[int]:
    v = str(val).strip().lower()
    if 'phd' in v or 'ph.d' in v or 'doctorate' in v: return 4
    if 'master' in v:   return 3
    if 'bachelor' in v: return 2
    if 'high school' in v or 'secondary' in v: return 1
    return None

edu_mapped = df['education_level'].apply(map_education)
unmapped = int(edu_mapped.isna().sum())
if unmapped:
    print(f'  ⚠ {unmapped} unmapped education values — filling with median')
    edu_mapped = edu_mapped.fillna(edu_mapped.median())
df['education_level_clean'] = edu_mapped.astype(float)
print('✔ education_level_clean  — ordinal 1–4')

# ── Gender encoding ──────────────────────────────────────────────────────────
gender_lower = df['gender'].str.strip().str.lower()
le = LabelEncoder()
df['gender_encoded'] = le.fit_transform(gender_lower).astype(int)
gender_map = dict(zip(gender_lower.unique().tolist(), df['gender_encoded'].unique().tolist()))
print(f'✔ gender_encoded         — {gender_map}')

# ── Seniority flag ───────────────────────────────────────────────────────────
kw = ['senior','director','manager','vp','chief','head','lead','principal']
df['is_senior'] = df['job_title'].str.lower().str.contains('|'.join(kw), regex=True).astype(int)
print(f'✔ is_senior              — {int(df["is_senior"].sum()):,} senior / {len(df):,} total')

# ── Job title dummies (top-15 + Other) ───────────────────────────────────────
top_jobs = df['job_title'].value_counts().nlargest(15).index.tolist()
df['job_category'] = df['job_title'].where(df['job_title'].isin(top_jobs), other='Other')
job_dummies = pd.get_dummies(df['job_category'], prefix='jobd', drop_first=False).astype(int)
df = pd.concat([df, job_dummies], axis=1)
print(f'✔ jobd_*                 — {job_dummies.shape[1]} one-hot columns')

# ── Interaction & polynomial features ────────────────────────────────────────
df['exp_x_edu']   = df['years_of_experience'] * df['education_level_clean']
df['age_squared'] = df['age'] ** 2
df['exp_squared'] = df['years_of_experience'] ** 2
print('✔ exp_x_edu, age_squared, exp_squared — interaction/polynomial features')

# ── Log-transform target ─────────────────────────────────────────────────────
df['log_salary'] = np.log1p(df['salary'])
print('✔ log_salary             — log1p(salary)')

print(f'\nFinal dataframe: {df.shape[0]:,} rows × {df.shape[1]} columns')
df[['age','years_of_experience','education_level_clean',
    'gender_encoded','is_senior','exp_x_edu','salary','log_salary']].describe().T

---
## Cell 4 — Visualizations

Eight inline charts exploring salary distributions, demographic breakdowns, role-based comparisons, and feature correlations. All charts are also saved to `./output_plots/` for the Word report.

In [ ]:
OUTPUT_DIR = 'output_plots'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_fig(name: str) -> None:
    plt.savefig(os.path.join(OUTPUT_DIR, name), dpi=150, bbox_inches='tight')

# ── 1. Salary distribution: raw vs log ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['salary'].to_numpy(), bins=50, color='#3b82d4', edgecolor='white')
axes[0].set_title('Salary Distribution (Raw)', fontweight='bold')
axes[0].set_xlabel('Salary (USD)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].hist(df['log_salary'].to_numpy(), bins=50, color='#7c5cd8', edgecolor='white')
axes[1].set_title('Salary Distribution (Log-transformed)', fontweight='bold')
axes[1].set_xlabel('log(Salary + 1)')
fig.suptitle('Figure 1 — Salary Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('01_salary_distribution.png')
plt.show()

# ── 2. Salary by education level ─────────────────────────────────────────────
edu_order_plot = ["high school", "bachelor's", "master's", "phd"]
plot_df = df.copy()
plot_df['edu_lower'] = plot_df['education_level'].str.strip().str.lower()
def to_short_edu(v):
    if 'phd' in v or 'ph.d' in v or 'doctorate' in v: return 'phd'
    if 'master' in v: return "master's"
    if 'bachelor' in v: return "bachelor's"
    if 'high school' in v or 'secondary' in v: return 'high school'
    return v
plot_df['edu_short'] = plot_df['edu_lower'].apply(to_short_edu)
edu_plot_data = plot_df[plot_df['edu_short'].isin(edu_order_plot)].copy()
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=edu_plot_data, x='edu_short', y='salary',
            order=edu_order_plot, palette='Blues', ax=ax)
ax.set_title('Figure 2 — Salary by Education Level', fontweight='bold')
ax.set_xlabel('Education Level')
ax.set_ylabel('Salary (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
save_fig('02_salary_by_education.png')
plt.show()

# ── 3. Salary by gender ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
sns.violinplot(data=df, x='gender', y='salary', palette='Set2', ax=ax, inner='quartile')
ax.set_title('Figure 3 — Salary by Gender', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
save_fig('03_salary_by_gender.png')
plt.show()

# ── 4. Experience vs Salary scatter ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(df['years_of_experience'].to_numpy(), df['salary'].to_numpy(),
                c=df['education_level_clean'].to_numpy(dtype=float),
                cmap='viridis', alpha=0.4, s=18, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Education Level (ordinal)')
ax.set_title('Figure 4 — Experience vs Salary', fontweight='bold')
ax.set_xlabel('Years of Experience')
ax.set_ylabel('Salary (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
save_fig('04_experience_vs_salary.png')
plt.show()

# ── 5. Top-15 job titles by median salary ─────────────────────────────────────
top15 = (df.groupby('job_title')['salary'].median()
           .nlargest(15).reset_index().sort_values('salary'))
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top15['job_title'].tolist(), top15['salary'].tolist(), color='#3b82d4')
ax.set_title('Figure 5 — Top 15 Job Titles by Median Salary', fontweight='bold')
ax.set_xlabel('Median Salary (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
save_fig('05_top15_jobs_salary.png')
plt.show()

# ── 6. Correlation heatmap ────────────────────────────────────────────────────
num_cols = ['age','years_of_experience','education_level_clean',
            'gender_encoded','is_senior','exp_x_edu','age_squared','exp_squared','salary']
corr = df[num_cols].corr()
mask = np.triu(np.ones(corr.shape, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, ax=ax)
ax.set_title('Figure 6 — Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
save_fig('06_correlation_heatmap.png')
plt.show()

# ── 7. Salary by experience band ──────────────────────────────────────────────
df['exp_bin'] = pd.cut(df['years_of_experience'], bins=[0,2,5,10,15,20,50],
                        labels=['0-2','3-5','6-10','11-15','16-20','20+'])
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x='exp_bin', y='salary', palette='Purples', ax=ax)
ax.set_title('Figure 7 — Salary by Experience Band', fontweight='bold')
ax.set_xlabel('Years of Experience')
ax.set_ylabel('Salary (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
save_fig('07_salary_by_exp_band.png')
plt.show()

# ── 8. Senior vs Non-Senior salary ───────────────────────────────────────────
plot_senior = df.copy()
plot_senior['seniority'] = df['is_senior'].map({0: 'Non-Senior', 1: 'Senior'})
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=plot_senior, x='seniority', y='salary',
            order=['Non-Senior','Senior'],
            palette={'Non-Senior':'#e5e7eb','Senior':'#3b82d4'}, ax=ax)
ax.set_title('Figure 8 — Salary: Senior vs Non-Senior', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Salary (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
save_fig('08_senior_vs_nonsenior.png')
plt.show()

print('✔ All 8 plots rendered inline and saved to ./output_plots/')

---
## Cell 5 — Model Training & Evaluation

Build a 24-feature matrix, split 80/20 (train/test), and train four scikit-learn pipelines (`StandardScaler` + model). Each model is evaluated with **5-fold cross-validation** on the train set, then scored on the held-out test set. All error metrics are back-transformed to USD via `expm1`.

The **Gradient Boosting** model is selected as the winner and used for feature-importance and residual diagnostics.

In [ ]:
# ── Feature matrix & target ───────────────────────────────────────────────────
feature_cols = (
    ['age','years_of_experience','education_level_clean',
     'gender_encoded','is_senior','exp_x_edu','age_squared','exp_squared']
    + [c for c in df.columns.tolist() if c.startswith('jobd_')]
)
X = df[feature_cols].astype(float)
y = df['log_salary']
print(f'Feature matrix : {X.shape[0]:,} rows × {X.shape[1]} features')
print(f'Target         : log_salary (back-transform with expm1 for USD metrics)')

# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,} samples  |  Test: {len(X_test):,} samples')

# ── Define models ─────────────────────────────────────────────────────────────
models = {
    'Linear Regression' : Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'Ridge Regression'  : Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=10.0))]),
    'Random Forest'     : Pipeline([('sc', StandardScaler()),
                                    ('m', RandomForestRegressor(n_estimators=200, max_depth=10,
                                                                random_state=42, n_jobs=-1))]),
    'Gradient Boosting' : Pipeline([('sc', StandardScaler()),
                                    ('m', GradientBoostingRegressor(n_estimators=200,
                                                                    learning_rate=0.05,
                                                                    max_depth=5,
                                                                    random_state=42))]),
}

# ── Train & evaluate ──────────────────────────────────────────────────────────
results = []
best_r2, best_name, best_pipe = -float('inf'), '', None

for name, pipe in models.items():
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    pipe.fit(X_train, y_train)
    log_pred = pipe.predict(X_test)
    pred_sal = np.expm1(log_pred)
    true_sal = np.expm1(np.array(y_test, dtype=float))
    mae  = float(mean_absolute_error(true_sal, pred_sal))
    rmse = float(mean_squared_error(true_sal, pred_sal)) ** 0.5
    r2   = float(r2_score(y_test, log_pred))
    results.append({'Model': name,
                    'CV R² (mean)': round(float(cv.mean()), 4),
                    'CV R² (±std)': f'±{cv.std():.4f}',
                    'Test R²': round(r2, 4),
                    'Test MAE ($)': f'${mae:,.0f}',
                    'Test RMSE ($)': f'${rmse:,.0f}'})
    if r2 > best_r2:
        best_r2, best_name, best_pipe = r2, name, pipe

# ── Benchmark table ───────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index('Model')
print('\n── 4-Model Benchmark Table ──────────────────────────────────────────────')
display(results_df.style.highlight_max(subset=['CV R² (mean)','Test R²'], color='#d6e4f0'))
print(f'\n★ Best model: {best_name}  (Test R² = {best_r2:.4f})')

assert best_pipe is not None

# ── Feature importances ───────────────────────────────────────────────────────
fi = best_pipe.named_steps['m'].feature_importances_
fi_df = (pd.Series(fi, index=feature_cols)
           .sort_values(ascending=False).head(20)
           .reset_index())
fi_df.columns = pd.Index(['Feature', 'Importance'])

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_df['Feature'].tolist()[::-1], fi_df['Importance'].tolist()[::-1], color='#3b82d4')
ax.set_title(f'Figure 9 — Top-20 Feature Importances ({best_name})', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
save_fig('09_feature_importances.png')
plt.show()

# ── Actual vs Predicted ───────────────────────────────────────────────────────
log_pred_best = best_pipe.predict(X_test)
pred_best = np.expm1(log_pred_best)
true_best = np.expm1(np.array(y_test, dtype=float))

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(true_best, pred_best, alpha=0.35, s=15, color='#3b82d4', edgecolors='none')
lims = [float(min(true_best.min(), pred_best.min())),
        float(max(true_best.max(), pred_best.max()))]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Salary (USD)')
ax.set_ylabel('Predicted Salary (USD)')
ax.set_title(f'Figure 10 — Actual vs Predicted ({best_name})', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
save_fig('10_actual_vs_predicted.png')
plt.show()

# ── Residual analysis ─────────────────────────────────────────────────────────
residuals = true_best - pred_best
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(residuals, bins=50, color='#7c5cd8', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Residual (USD)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].scatter(pred_best, residuals, alpha=0.3, s=12, color='#7c5cd8', edgecolors='none')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Predicted')
axes[1].set_xlabel('Predicted Salary (USD)')
axes[1].set_ylabel('Residual (USD)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
fig.suptitle(f'Figure 11 — Residual Analysis ({best_name})', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('11_residuals.png')
plt.show()

# ── Final summary ─────────────────────────────────────────────────────────────
mae_f  = float(mean_absolute_error(true_best, pred_best))
rmse_f = float(mean_squared_error(true_best, pred_best)) ** 0.5
print('\n' + '='*60)
print('PROJECT COMPLETE')
print(f'  Best Model : {best_name}')
print(f'  Test R²    : {best_r2:.4f}')
print(f'  Test MAE   : ${mae_f:,.0f}')
print(f'  Test RMSE  : ${rmse_f:,.0f}')
print('='*60)